# 05. Operaciones entre señales, cuidando los índices

**Objetivos de aprendizaje.** Al terminar podrás:

1. **Alinear** dos señales con distintos índices para poder sumarlas o multiplicarlas.
2. **Desplazar** e **invertir** una señal en el tiempo, y combinar ambas operaciones en el orden correcto.
3. **Submuestrear** y **sobremuestrear** (escalamiento en el tiempo) y relacionarlo con Nyquist.
4. Descomponer una señal en su parte **par** e **impar**.
5. Aplicar estas operaciones a audio.

**Requisito:** notebooks 00 y 02.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. El problema: sumar señales con distintos índices

$$x_1[n] = \{1, 2, \underset{\uparrow}{3}, 2, 1\} \;(n=-2..2) \qquad x_2[n] = \{\underset{\uparrow}{1}, 1, 1, 1\} \;(n = 0..3)$$

Queremos $y[n] = x_1[n] + x_2[n]$. Si sumamos los arreglos "tal cual":

In [ ]:
n1 = np.arange(-2, 3)
x1 = np.array([1.0, 2.0, 3.0, 2.0, 1.0])
n2 = np.arange(0, 4)
x2 = np.array([1.0, 1.0, 1.0, 1.0])

try:
    y = x1 + x2
except ValueError as error:
    print("ERROR:", error)

Python se queja porque los tamaños son distintos. Pero aunque fueran del mismo tamaño, **estaría mal**:
`x1[0]` es el instante $n=-2$ y `x2[0]` es el instante $n=0$: estaríamos sumando instantes diferentes.

### La solución: alinear sobre un eje común

**Paso 1.** El eje común va del índice más pequeño al más grande de las dos señales.

In [ ]:
n_min = min(n1[0], n2[0])
n_max = max(n1[-1], n2[-1])
n = np.arange(n_min, n_max + 1)
print("eje común n =", n)

**Paso 2.** Creamos señales de ceros con ese tamaño y **copiamos** cada valor en su lugar.
El instante $n_1[i]$ va en la posición `n1[i] - n_min` del nuevo arreglo.

In [ ]:
y1 = np.zeros(len(n))
y2 = np.zeros(len(n))

for i in range(len(n1)):
    posicion = n1[i] - n_min
    y1[posicion] = x1[i]
    print(f"  x1 en n = {n1[i]:2d}  ->  posición {posicion}")

for i in range(len(n2)):
    posicion = n2[i] - n_min
    y2[posicion] = x2[i]

print("n  =", n)
print("y1 =", y1)
print("y2 =", y2)

**Paso 3.** Con las señales alineadas, ya podemos sumar muestra a muestra.

In [ ]:
y = y1 + y2

figura, ejes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
pds.graficar_discreta(n1, x1, titulo="x1[n]", ax=ejes[0])
pds.graficar_discreta(n2, x2, titulo="x2[n]", ax=ejes[1])
pds.graficar_discreta(n, y, titulo="x1[n] + x2[n]", ax=ejes[2], color="C3")
for eje in ejes:
    eje.set_xlim(-3, 4)
plt.tight_layout()
plt.show()

Esto es lo que hacen `pds.alinear`, `pds.sumar_senales` y `pds.multiplicar_senales`:

In [ ]:
n_s, y_s = pds.sumar_senales(n1, x1, n2, x2)
pds.comprobar_iguales(y_s, y, "pds.sumar_senales da lo mismo")

n_p, y_p = pds.multiplicar_senales(n1, x1, n2, x2)
print("producto:", y_p, "  (sólo es distinto de cero donde AMBAS lo son)")

## 2. Desplazamiento: $y[n] = x[n-k]$

$y[n] = x[n-k]$ significa: "el valor que $x$ tenía en el instante $m$, ahora $y$ lo tiene en el instante $m + k$".

- $k > 0$: **retraso** → la señal se mueve a la **derecha**.
- $k < 0$: **adelanto** → la señal se mueve a la **izquierda**.

> Los **valores no cambian**. Sólo cambia el arreglo de **índices**: `n_nuevo = n + k`.

In [ ]:
n = np.arange(0, 4)
x = np.array([1.0, 2.0, 3.0, 4.0])

n_ret, x_ret = pds.desplazar(n, x, 2)        # x[n - 2]
n_ade, x_ade = pds.desplazar(n, x, -3)       # x[n + 3]

print("x[n]   : n =", n, "  valores =", x)
print("x[n-2] : n =", n_ret, "  valores =", x_ret)
print("x[n+3] : n =", n_ade, "  valores =", x_ade)

figura, ejes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
pds.graficar_discreta(n, x, titulo="x[n]", ax=ejes[0])
pds.graficar_discreta(n_ret, x_ret, titulo="x[n-2]: retraso (derecha)", ax=ejes[1])
pds.graficar_discreta(n_ade, x_ade, titulo="x[n+3]: adelanto (izquierda)", ax=ejes[2])
for eje in ejes:
    eje.set_xlim(-4, 6)
plt.tight_layout()
plt.show()

## 3. Inversión en el tiempo: $y[n] = x[-n]$

El valor en el instante $m$ pasa al instante $-m$: la señal se **refleja** respecto a $n = 0$.

**Paso 1:** cambiamos el signo de los índices. **Paso 2:** como el eje queda de mayor a menor, invertimos el orden de ambos arreglos.

In [ ]:
n_inv, x_inv = pds.invertir_tiempo(n, x)
print("x[-n]  : n =", n_inv, "  valores =", x_inv)

## 4. Combinar desplazamiento e inversión: $y[n] = x[-n + 2]$

Es la operación que más errores causa (y es la base de la convolución). Escribimos $x[-n+2] = x[-(n-2)]$:

1. Primero **invertimos**: $v[n] = x[-n]$.
2. Luego **desplazamos** $v$: $v[n-2] = x[-(n-2)] = x[-n+2]$ → se mueve **2 a la derecha**.

Comprobemos con un valor: $y[n] = x[-n+2]$ en $n = 2$ vale $x[0] = 1$. En $n = -1$ vale $x[3] = 4$.

In [ ]:
n_v, v = pds.invertir_tiempo(n, x)          # Paso 1: x[-n]
n_y, y = pds.desplazar(n_v, v, 2)           # Paso 2: desplazar +2

print("y[n] = x[-n+2] : n =", n_y, "  valores =", y)
pds.comprobar(y[n_y == 2][0] == 1, "y[2] = x[0] = 1")
pds.comprobar(y[n_y == -1][0] == 4, "y[-1] = x[3] = 4")

# El orden INCORRECTO: primero desplazar 2 y luego invertir da x[-n-2]
n_w, w = pds.desplazar(n, x, 2)
n_mal, mal = pds.invertir_tiempo(n_w, w)

figura, ejes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
pds.graficar_discreta(n, x, titulo="x[n]", ax=ejes[0])
pds.graficar_discreta(n_y, y, titulo="x[-n+2]  (invertir y luego desplazar)", ax=ejes[1], color="C2")
pds.graficar_discreta(n_mal, mal, titulo="INCORRECTO: desplazar y luego invertir = x[-n-2]", ax=ejes[2], color="C3")
for eje in ejes:
    eje.set_xlim(-6, 6)
plt.tight_layout()
plt.show()

> **Un método que siempre funciona:** para saber dónde queda cada valor en $y[n] = x[a n + b]$, despeja: el valor $x[m]$ aparece
> donde $a n + b = m$, es decir en $n = (m - b)/a$.

## 5. Escalamiento en el tiempo: submuestreo y sobremuestreo

### Submuestreo (diezmado) por $M$: $y[n] = x[Mn]$
Nos quedamos con una de cada $M$ muestras. Si $x$ tenía $f_s$, $y$ tiene $f_s/M$.
Su frecuencia digital se **multiplica** por $M$, así que puede dejar de cumplir Nyquist.

In [ ]:
n, x = pds.senoidal_digital_muestras(N=40, f=0.05)
n_d, x_d = pds.submuestrear(n, x, 2)

figura, ejes = plt.subplots(1, 2, figsize=(13, 3))
pds.graficar_discreta(n, x, titulo="x[n], f = 0.05", ax=ejes[0])
pds.graficar_discreta(n_d, x_d, titulo="x[2n]: f = 0.1 (oscila el doble de rápido)", ax=ejes[1])
plt.tight_layout()
plt.show()

### Sobremuestreo (expansión) por $L$
Se insertan $L-1$ ceros entre muestras (después se "rellenan" con un filtro: interpolación).

In [ ]:
n = np.arange(0, 5)
x = np.array([1.0, 2.0, 3.0, 2.0, 1.0])
n_e, x_e = pds.sobremuestrear(n, x, 3)

figura, ejes = plt.subplots(1, 2, figsize=(13, 3))
pds.graficar_discreta(n, x, titulo="x[n]", ax=ejes[0])
pds.graficar_discreta(n_e, x_e, titulo="sobremuestreo por L = 3", ax=ejes[1])
plt.tight_layout()
plt.show()

## 6. Parte par y parte impar

$$x_{par}[n] = \frac{x[n] + x[-n]}{2} \qquad x_{impar}[n] = \frac{x[n] - x[-n]}{2} \qquad x = x_{par} + x_{impar}$$

In [ ]:
n, u = pds.escalon(-6, 6)
n_c, par, impar = pds.parte_par_impar(n, u)

figura, ejes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
pds.graficar_discreta(n, u, titulo="u[n]", ax=ejes[0])
pds.graficar_discreta(n_c, par, titulo="parte par", ax=ejes[1])
pds.graficar_discreta(n_c, impar, titulo="parte impar", ax=ejes[2])
plt.tight_layout()
plt.show()

pds.comprobar_iguales(par + impar, u, "x = x_par + x_impar")

## 7. Operaciones con audio

Con audio, los índices empiezan en 0 y todas las señales tienen la misma $f_s$, así que basta con igualar los tamaños.
Ejemplo: **fade-out** (desvanecer) = multiplicar por una rampa descendente.

In [ ]:
fs, x = pds.cargar_audio("../datos/tono_440hz.wav")
N = len(x)
rampa_bajada = np.linspace(1, 0, N)       # de 1 a 0 en N muestras
y = x * rampa_bajada                      # producto muestra a muestra

t = np.arange(N) / fs
pds.graficar_continua(t, y, titulo="Fade-out: x[n]·rampa[n]")
plt.show()
pds.reproducir(y, fs)

## Comprueba tu comprensión

1. Si $x[n]$ es distinta de cero sólo en $0 \le n \le 4$, ¿dónde es distinta de cero $x[n+2]$? ¿Y $x[-n]$? ¿Y $x[3-n]$?
2. ¿Por qué no podemos hacer `x1 + x2` directamente si tienen distintos índices?
3. Un tono de $f = 0.2$ se submuestrea por 3. ¿Cuál es la nueva frecuencia digital? ¿Hay aliasing?

<details><summary>Ver respuestas</summary>

1. $-2 \le n \le 2$; $-4 \le n \le 0$; $-1 \le n \le 3$.
2. Porque la posición $i$ de cada arreglo corresponde a instantes distintos (y pueden tener distinto tamaño).
3. $0.6 > 0.5$ → hay aliasing; se ve como $1 - 0.6 = 0.4$.
</details>